In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv('jamb_exam_results.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [7]:
if 'student_id' in df.columns:
    df = df.drop('student_id', axis=1)
df = df.fillna(0)
y = df.jamb_score.values
X = df.drop('jamb_score', axis=1)

# train / val / test (60/20/20)
df_full_train, df_test, y_full_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
df_train, df_val, y_train, y_val = train_test_split(df_full_train, y_full_train, test_size=0.25, random_state=1)
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))


In [9]:
#Вопрос 1
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

feature = dv.feature_names_[dt.tree_.feature[0]]
print("Признак для разбиения:", feature)

Признак для разбиения: study_hours_per_week


In [12]:
#Вопрос 2
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
print("RMSE:", round(rmse, 2))

RMSE: 42.14


In [27]:
#Вопрос 3
results = []

for n in range(10, 210, 10):
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    results.append((n, rmse))

# --- создаём таблицу результатов
df_scores = pd.DataFrame(results, columns=['n_estimators', 'rmse'])

df_scores['diff'] = df_scores.rmse.diff().round(3)
plateau = df_scores.loc[df_scores['diff'].abs() < 0.001]

if len(plateau) > 0:
    best_n = plateau.iloc[0].n_estimators

else:
    best_n = df_scores.loc[df_scores.rmse.idxmin(), 'n_estimators']

print("RMSE перестаёт улучшаться после n_estimators =", int(best_n))
print(df_scores)


RMSE перестаёт улучшаться после n_estimators = 90
    n_estimators       rmse   diff
0             10  42.137242    NaN
1             20  41.461215 -0.676
2             30  41.106171 -0.355
3             40  40.917194 -0.189
4             50  40.852279 -0.065
5             60  40.784281 -0.068
6             70  40.677098 -0.107
7             80  40.539333 -0.138
8             90  40.504346 -0.035
9            100  40.516805  0.012
10           110  40.593353  0.077
11           120  40.624850  0.031
12           130  40.650841  0.026
13           140  40.594852 -0.056
14           150  40.596715  0.002
15           160  40.603508  0.007
16           170  40.627546  0.024
17           180  40.641314  0.014
18           190  40.631355 -0.010
19           200  40.601019 -0.030


In [28]:
#Вопрос 4
depths = [10, 15, 20, 25]
mean_rmse = []

for d in depths:
    rmse_list = []
    for n in range(10, 210, 10):
        rf = RandomForestRegressor(max_depth=d, n_estimators=n, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        mse = mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mse)
        rmse_list.append(rmse)
    mean_rmse.append((d, np.mean(rmse_list)))

df_depths = pd.DataFrame(mean_rmse, columns=['max_depth', 'mean_rmse'])
best_depth = df_depths.loc[df_depths.mean_rmse.idxmin(), 'max_depth']
print("Лучшее значение max_depth:", best_depth)
display(df_depths)


Лучшее значение max_depth: 10


,max_depth,mean_rmse
0,10,40.392498
1,15,40.735282
2,20,40.739734
3,25,40.787866


In [30]:
#Вопрос 5
rf_final = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_final.fit(X_train, y_train)

importances = pd.Series(rf_final.feature_importances_, index=dv.feature_names_)
importances = importances.sort_values(ascending=False)
print("Самый важный признак:", importances.index[0])
display(importances.head(10))

Самый важный признак: study_hours_per_week


,0
study_hours_per_week,0.248354
attendance_rate,0.149729
distance_to_school,0.136486
teacher_quality,0.082682
age,0.069311
assignments_completed,0.031517
socioeconomic_status=High,0.025714
parent_involvement=High,0.022919
it_knowledge=High,0.017719
parent_education_level=Secondary,0.016957


In [34]:
# Вывод ответов по вопросам

print("1)",feature)
print("2) 42.14")
print("3)", int(best_n))
print("4)", best_depth)
print("5)", importances.index[0])


1) study_hours_per_week
2) 42.14
3) 90
4) 10
5) study_hours_per_week
